# Baseline ML Classification | A/B Test

In [1]:
import pandas as pd, json, joblib, os
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/gold/demo_cases.csv')
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['expected_label'], test_size=0.2, random_state=42)


In [2]:
models = {
    'LinearSVC': Pipeline([('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2))),
                           ('clf', LinearSVC(C=1.0, max_iter=10000))]),
    'LogReg': Pipeline([('tfidf', TfidfVectorizer()), ('clf', LogisticRegression())]),
    'NaiveBayes': Pipeline([('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())]),
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    f1 = f1_score(y_test, pred, average='weighted')
    results[name] = f1
    print(f"{name} F1: {f1}")

print("A/B comparison table:\n", results)

LinearSVC F1: 0.35714285714285715
LogReg F1: 0.12698412698412698
NaiveBayes F1: 0.12698412698412698
A/B comparison table:
 {'LinearSVC': 0.35714285714285715, 'LogReg': 0.12698412698412698, 'NaiveBayes': 0.12698412698412698}


In [3]:
best = models['LinearSVC']
os.makedirs('../artifacts', exist_ok=True)
joblib.dump(best, '../artifacts/baseline_best_model.joblib')
metrics = {'model':'LinearSVC','f1': results['LinearSVC']}
json.dump(metrics, open('../artifacts/baseline_metrics.json','w'))

try:
    import mlflow
    print("MLFlow would log:", metrics)
except:
    print("MLflow not configured, skipping")

MLFlow would log: {'model': 'LinearSVC', 'f1': 0.35714285714285715}


## Висновки
A/B результати: LinearSVC F1=X.XXXX, найкраща модель.
Збережено в artifacts/baseline_best_model.joblib.